# Comparative experiment for GCNconv Bubble Resolution net on different coverages level

In [10]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import json, math, time, random, re, glob, os, itertools
import numpy as np
import pandas as pd

USE_DIRECTML = False

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

import matplotlib.pyplot as plt

In [11]:
NUC2ID = {'A':1, 'C':2, 'G':3, 'T':4, 'N':5}

def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def tokens_to_onehot(tokens: torch.LongTensor, num_classes: int = 6):
    N, K = tokens.shape
    classes = torch.arange(num_classes, device=tokens.device).view(1, 1, num_classes)
    return (tokens.unsqueeze(-1) == classes).float()

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())


In [12]:
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data

class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset with token sequences for CNN encoding.
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float]=None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        for n in _safe_get(rec,"nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"upstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"downstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"inside_nodes",[]):
            if isinstance(n,dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov"))

        ensure_node(start_seq)
        ensure_node(end_seq)

        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec,"edges",[]):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes",0)),
                float(e.get("len_bp",0)),
                float(e.get("cov_min",0)),
                float(e.get("cov_mean",0.0)),
                1.0 if e.get("on_ref",False) else 0.0
            ])

        node_order = [None]*len(node_seqs)
        for s,i in node_seqs.items():
            node_order[i]=s

        # store tokens only, CNN handles one-hot
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)

        start_idx = torch.tensor(node_seqs[start_seq])
        end_idx   = torch.tensor(node_seqs[end_seq])

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.zeros((0,5))

        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1

        paths_list = _safe_get(rec,"paths",[])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list):
            label_path_idx = lp
            if num_edges > 0:
                y_edge_mask[torch.tensor(paths_list[lp], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,
            x_cov=x_cov,
            edge_index=edge_index,
            edge_attr=edge_attr,
            start_idx=start_idx,
            end_idx=end_idx,
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,
            label_path_idx=torch.tensor(label_path_idx),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"])
        if "k" in rec:
            data.k = torch.tensor(rec["k"])
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])

In [13]:
from pathlib import Path
from typing import Dict, List

SEARCH_DIRS = [
    Path("lower_cov_data"),
    Path("other_cov_data"),
]

def discover_base_datasets_for_cov(coverage: int) -> Dict[str, str]:
    """
    Znajdź wszystkie pliki w formacie:
        {genome_id}_dataset_cov{coverage}_ratio_lt_3.5.jsonl
    w katalogach z SEARCH_DIRS.

    Zwraca słownik:
        {genome_id: pełna_ścieżka_do_pliku}
    """
    pattern = f"*dataset_k21_cov{coverage}_ratio_lt_3.5.jsonl"
    candidates: List[Path] = []

    for d in SEARCH_DIRS:
        if not d.is_dir():
            continue
        candidates.extend(d.glob(pattern))

    unique = sorted({p.resolve() for p in candidates})

    if not unique:
        raise FileNotFoundError(
            f"Nie znaleziono plików dla pokrycia {coverage} (wzorzec: {pattern}) "
            f"w katalogach: {', '.join(str(d) for d in SEARCH_DIRS)}"
        )

    mapping: Dict[str, str] = {}
    for p in unique:
        name = p.name
        genome_id = name.split("_dataset_cov")[0]
        mapping[genome_id] = str(p)

    print(
        f"[discover_base_datasets_for_cov] coverage={coverage} -> "
        f"znaleziono {len(mapping)} genomów:"
    )
    for gid in sorted(mapping.keys()):
        print("  ", gid)

    return mapping

# pokrycia do nowego eksperymentu
COVERAGES = [10, 20, 30]

COV_TO_FILES = {cov: discover_base_datasets_for_cov(cov) for cov in COVERAGES}

# jeśli chcesz zobaczyć wszystkie genomy per coverage:
for cov in COVERAGES:
    print(f"coverage={cov} -> genomów: {sorted(COV_TO_FILES[cov].keys())}")

# wspólny zbiór genomów (jeśli potrzebny)
COMMON_GENOMES = sorted(set.intersection(*[set(m.keys()) for m in COV_TO_FILES.values()]))
print("COMMON_GENOMES (występują dla wszystkich pokryć):", COMMON_GENOMES)


def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)


[discover_base_datasets_for_cov] coverage=10 -> znaleziono 3 genomów:
   ecoli_dataset_k21_cov10_ratio_lt_3.5.jsonl
   klebsiella_dataset_k21_cov10_ratio_lt_3.5.jsonl
   proteus_dataset_k21_cov10_ratio_lt_3.5.jsonl
[discover_base_datasets_for_cov] coverage=20 -> znaleziono 3 genomów:
   ecoli_dataset_k21_cov20_ratio_lt_3.5.jsonl
   klebsiella_dataset_k21_cov20_ratio_lt_3.5.jsonl
   proteus_dataset_k21_cov20_ratio_lt_3.5.jsonl
[discover_base_datasets_for_cov] coverage=30 -> znaleziono 3 genomów:
   ecoli_dataset_k21_cov30_ratio_lt_3.5.jsonl
   klebsiella_dataset_k21_cov30_ratio_lt_3.5.jsonl
   proteus_dataset_k21_cov30_ratio_lt_3.5.jsonl
coverage=10 -> genomów: ['ecoli_dataset_k21_cov10_ratio_lt_3.5.jsonl', 'klebsiella_dataset_k21_cov10_ratio_lt_3.5.jsonl', 'proteus_dataset_k21_cov10_ratio_lt_3.5.jsonl']
coverage=20 -> genomów: ['ecoli_dataset_k21_cov20_ratio_lt_3.5.jsonl', 'klebsiella_dataset_k21_cov20_ratio_lt_3.5.jsonl', 'proteus_dataset_k21_cov20_ratio_lt_3.5.jsonl']
coverage=30 -> 

In [14]:
class HyperbubbleGNN(nn.Module):
    """
    CNN-based sequence encoder + 2× GCNConv + MLP edge classifier.
    Designed to be DirectML-friendly (no dense scatter ops).
    """
    def __init__(
        self,
        vocab_size=6,        # one-hot channels
        cnn_channels=32,     # Conv1d output channels
        gcn_hidden=64,
        edge_hidden=64,
        edge_feat_dim=5,
        dropout=0.0,
    ):
        super().__init__()

        # 1-hot CNN encoder over k-mer tokens
        self.cnn = nn.Conv1d(vocab_size, cnn_channels, kernel_size=3, padding=1)

        # node feature = CNN embedding + coverage
        self.node_in = cnn_channels + 1

        # sparse GCN layers (operate directly on edge_index)
        self.gcn1 = GCNConv(self.node_in, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.gcn_hidden = gcn_hidden

        # edge classifier MLP
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] (Long, k-mer tokens)
        x_cov:      [N, 1] (Float, node coverage)
        returns:    [N, cnn_channels + 1]
        """
        # NOTE: tokens_to_onehot should be implemented in a DML-safe way
        # e.g. using broadcasting instead of F.one_hot.
        onehot = tokens_to_onehot(seq_tokens)      # [N, K, vocab_size]
        x = onehot.permute(0, 2, 1)                # [N, vocab_size, K]

        H = F.relu(self.cnn(x))                    # [N, C, K]
        H = H.mean(dim=2)                          # [N, C] global pooling

        return torch.cat([H, x_cov], dim=1)        # [N, C+1]

    def forward(self, data):
        """
        data:
          - seq_tokens [N,K]
          - x_cov      [N,1]
          - edge_index [2,E]
          - edge_attr  [E,5] (optional)
          - batch      [N]   (optional, for multi-graph batches)
        returns:
          - logits [E]
        """
        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)  # [N, node_in]

        # GCN over whole batch (PyG handles 'batch' internally if present)
        H = F.relu(self.gcn1(X0, data.edge_index))
        H = self.dropout(H)

        H = F.relu(self.gcn2(H, data.edge_index))
        H = self.dropout(H)

        # edge-wise features
        u, v = data.edge_index
        U = H[u]
        V = H[v]

        if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0:
            EA = data.edge_attr
        else:
            E = data.edge_index.size(1)
            EA = H.new_zeros((E, 5))

        feats = torch.cat([U, V, EA], dim=1)       # [E, 2*gcn_hidden + 5]
        logits = self.edge_mlp(feats).squeeze(-1)  # [E]

        return logits

In [15]:
def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Return list of (cand_idx_tensor, gold_pos_tensor) for EVERY decision node
    whose source node appears on the labeled path. This is orientation-agnostic.
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

def train_one_epoch_choice_all(model, loader, optim, device):
    model.train()
    total_loss, total_decisions = 0.0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        batch_loss = 0.0
        dec_used = 0
        batch_correct = 0        # NEW
        batch_total = 0          # NEW

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            for cand_idx, gold_pos in decisions:

                # compute loss
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                batch_loss += ce
                dec_used += 1

                # compute train-time accuracy
                pred = logits[cand_idx].argmax().item()
                gold = gold_pos.item()
                batch_correct += int(pred == gold)
                batch_total += 1

        if dec_used == 0:
            continue

        batch_loss = batch_loss / dec_used
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss += batch_loss.item()
        total_decisions += dec_used

        # ---- PRINT TRAIN BATCH ACCURACY ----
        if batch_total > 0:
            acc = batch_correct / batch_total
            print(f"[train] batch_decisions={batch_total} | batch_acc={acc:.3f}")

    return total_loss / max(total_decisions, 1)


def _cpu_state_dict(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

best_ckpt = {
    "epoch": 0,
    "dec_acc": -1.0,
    "bub_acc": -1.0,
    "state_dict": None,
}

def _is_better(bub_acc, dec_acc, best):
    if bub_acc > best["bub_acc"]:
        return True
    if bub_acc == best["bub_acc"] and dec_acc > best["dec_acc"]:
        return True
    return False

@torch.no_grad()
def eval_choice_all(model, loader, device):
    model.eval()
    total_decisions, correct_decisions = 0, 0
    total_bubbles, correct_bubbles = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue
            total_bubbles += 1

            all_correct = True
            for cand_idx, gold_pos in decisions:
                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                correct_decisions += ok
                total_decisions += 1
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    dec_acc = correct_decisions / max(total_decisions, 1)
    bub_acc = correct_bubbles / max(total_bubbles, 1)
    print(f"[choice-eval] decisions={total_decisions} acc_dec={dec_acc:.3f} | bubbles={total_bubbles} acc_bub={bub_acc:.3f}")
    return dec_acc, bub_acc

In [16]:
SEED = random.randint(1,100)
BATCH_SIZE = 64
NUM_WORKERS = 0
EPOCHS = 200
PATIENCE = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
VAL_FRAC = 0.1

PRINT_EVERY = 50

In [17]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def split_train_val(n: int, val_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_val = int(round(val_frac * n))
    val_idx = order[:n_val]
    tr_idx  = order[n_val:]
    return tr_idx.tolist(), val_idx.tolist()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Zwraca listę (cand_idx, gold_pos) dla każdego węzła źródłowego w grafie g,
    dla którego istnieje decyzja (zaznaczone krawędzie y_edge_mask==1).
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

@torch.no_grad()
def _candidate_sets(data, logits: torch.Tensor):
    """
    Zwraca listę (cand_idx [M], gold_pos [1], g) dla każdej decyzji w batchu.
    """
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)
    out = []
    for g in range(num_graphs):
        decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in decisions:
            out.append((cand_idx, gold_pos, g))
    return out

def decision_ce_loss_from_batch(logits: torch.Tensor, data, reduction: str = "mean"):
    """
    Cross-entropy liczona po decyzjach (candidate sets).
    """
    if logits.numel() == 0:
        return logits.new_zeros(()), 0
    losses, decisions = [], 0
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)

    for g in range(num_graphs):
        cands = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in cands:
            cand_logits = logits[cand_idx]     # [M]
            target = gold_pos.view(1)          # [1]
            losses.append(F.cross_entropy(cand_logits.unsqueeze(0), target))
            decisions += 1

    if decisions == 0:
        return logits.new_zeros(()), 0
    loss = torch.stack(losses)
    return (loss.mean() if reduction == "mean" else loss.sum()), decisions


@torch.no_grad()
def eval_rich_metrics(model, loader, device) -> Dict[str, float]:
    """
    Minimalny zestaw metryk:
      - acc_dec   : dokładność decyzji (wybór ścieżki)
      - acc_bub   : dokładność bąbli (wszystkie decyzje w bąblu poprawne)
      - precision_dec, recall_dec, f1_dec : metryki binarne dla decyzji
      - brier     : błąd Briera dla prawdopodobieństwa poprawnej ścieżki

    Wszystko w oparciu o candidate sets, bez ECE / ROC / PR-AUC / MRR.
    """
    model.eval()
    dec_correct, dec_total = 0, 0
    bubble_ok = {}  # key: (id(data), g)

    confs_gold = []   # p(model poprawny) dla poprawnej ścieżki (gold)
    labels_gold = []  # 1/0 czy decyzja faktycznie poprawna

    y_true_dec = []   # 1 jeśli decyzja poprawna
    y_pred_dec = []   # 1 jeśli model jest "pewny" (p(pred) >= 0.5)

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        for cand_idx, gold_pos, g in _candidate_sets(data, logits):
            cand_logits = logits[cand_idx]              # [M]
            probs = F.softmax(cand_logits, dim=0)
            pred = int(torch.argmax(cand_logits).item())
            gold = int(gold_pos.item())

            ok = int(pred == gold)
            dec_total += 1
            dec_correct += ok

            key = (id(data), int(g))
            bubble_ok.setdefault(key, True)
            bubble_ok[key] = bubble_ok[key] and bool(ok)

            # Brier: prawdopodobieństwo poprawnej ścieżki
            conf_gold = float(probs[gold].item())
            confs_gold.append(conf_gold)
            labels_gold.append(float(ok))

            # klasyfikacja binarna dla precision/recall/F1:
            #   true = decyzja poprawna
            #   pred = pewny (p(pred) >= 0.5)
            y_true_dec.append(float(ok))
            conf_pred = float(probs[pred].item())
            y_pred_dec.append(1.0 if conf_pred >= 0.5 else 0.0)

    acc_dec = dec_correct / max(dec_total, 1)

    bub_total = len(bubble_ok)
    bub_correct = sum(1 for ok in bubble_ok.values() if ok)
    acc_bub = bub_correct / max(bub_total, 1)

    # Brier
    confs_gold = np.array(confs_gold, dtype=np.float64)
    labels_gold = np.array(labels_gold, dtype=np.float64)
    if confs_gold.size:
        brier = float(((confs_gold - labels_gold) ** 2).mean())
    else:
        brier = float("nan")

    # precision / recall / F1 dla decyzji
    y_true_dec = np.array(y_true_dec, dtype=np.int32)
    y_pred_dec = np.array(y_pred_dec, dtype=np.int32)
    if y_true_dec.size:
        tp = int(((y_true_dec == 1) & (y_pred_dec == 1)).sum())
        fp = int(((y_true_dec == 0) & (y_pred_dec == 1)).sum())
        fn = int(((y_true_dec == 1) & (y_pred_dec == 0)).sum())

        precision_dec = tp / max(tp + fp, 1)
        recall_dec    = tp / max(tp + fn, 1)
        if precision_dec + recall_dec > 0:
            f1_dec = 2 * precision_dec * recall_dec / (precision_dec + recall_dec)
        else:
            f1_dec = 0.0
    else:
        precision_dec = recall_dec = f1_dec = float("nan")

    return {
        "acc_dec": acc_dec,
        "acc_bub": acc_bub,
        "precision_dec": precision_dec,
        "recall_dec": recall_dec,
        "f1_dec": f1_dec,
        "brier": brier,
        "decisions": dec_total,
        "bubbles": bub_total,
    }


def train_one_epoch(model, loader, device, optimizer, grad_clip: Optional[float] = 1.0):
    model.train()
    total_loss, total_decisions = 0.0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(data)
        loss, ndec = decision_ce_loss_from_batch(logits, data, reduction="mean")
        if ndec == 0:
            continue
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += float(loss.item()) * ndec
        total_decisions += ndec
    return total_loss / max(total_decisions, 1)


def fit_variant(build_model_fn,
                train_loader,
                val_loader,
                device,
                out_ckpt: Path,
                epochs=20,
                lr=1e-3,
                weight_decay=0,
                patience=5,
                grad_clip=1.0,
                seed=42,
                print_every=5,
                early_stop=True):
    model = build_model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = -1.0
    best_state = None
    waited = 0
    history = []

    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, device, opt, grad_clip)
        val_stats = eval_rich_metrics(model, val_loader, device)
        history.append({"epoch": ep, "train_loss": tr_loss, **val_stats})
        metric = val_stats["acc_bub"]

        if (ep % print_every == 0) or (ep == 1):
            print(
                f"[ep {ep:03d}] train_loss={tr_loss:.4f}  "
                f"val_bub={val_stats['acc_bub']:.4f}  "
                f"val_dec={val_stats['acc_dec']:.4f}"
            )

        if metric > best_val:
            best_val = metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1

        if early_stop and waited >= patience:
            break

    if best_state is None:
        best_state = model.state_dict()
    torch.save(best_state, out_ckpt)
    return best_state, pd.DataFrame(history)


In [18]:
COVERAGE_SWEEP = 20
GENOME_TO_PATH = COV_TO_FILES[COVERAGE_SWEEP]
GENOMES = sorted(GENOME_TO_PATH.keys())
print("Genomy używane w sweepie architektur:", GENOMES)

OUTDIR = Path("coverage_sweep_28_12")
OUTDIR.mkdir(parents=True, exist_ok=True)

Genomy używane w sweepie architektur: ['ecoli_dataset_k21_cov20_ratio_lt_3.5.jsonl', 'klebsiella_dataset_k21_cov20_ratio_lt_3.5.jsonl', 'proteus_dataset_k21_cov20_ratio_lt_3.5.jsonl']


In [19]:
import pandas as pd

REPEATS = 5

# STAŁA ARCHITEKTURA – UZUPEŁNIJ TO LICZBAMI Z POPRZEDNIEGO SWEEPA
BEST_EMB = 8      # <- wstaw swoje
BEST_GCN = 16     # <- wstaw swoje
BEST_EDGE = 256   # <- wstaw swoje
USE_LSTM = False  # <- jeśli masz wersję LSTM, ustaw True/False

variant = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"
print("Uruchamianie eksperymentu coverage-sweep dla wariantu:", variant)

SUMMARY_CSV = OUTDIR / "coverage_sweep_results.csv"

# nagłówek CSV (jeśli plik nie istnieje)
if not SUMMARY_CSV.exists():
    pd.DataFrame(columns=[
        "variant",
        "coverage",
        "runs",        # liczba powtórzeń na fold (REPEATS)
        "params",
        "acc_dec",
        "precision_dec",
        "recall_dec",
        "f1_dec",
        "acc_bub",
        "decisions",
        "bubbles",
    ]).to_csv(SUMMARY_CSV, index=False)

set_seed(SEED)

def build_model():
    return HyperbubbleGNN(
        vocab_size=6,
        cnn_channels=BEST_EMB,
        gcn_hidden=BEST_GCN,
        edge_hidden=BEST_EDGE,
        edge_feat_dim=5,
        dropout=0.0,
    )

for cov in COVERAGES:
    GENOME_TO_PATH = COV_TO_FILES[cov]

    # jeśli chcesz wymusić dokładnie 3 genomy wspólne dla wszystkich pokryć:
    # GENOMES = COMMON_GENOMES
    # a jeśli tylko dla tego pokrycia, bierz po prostu wszystkie:
    GENOMES = sorted(GENOME_TO_PATH.keys())

    print(f"\n=== COVERAGE SWEEP: coverage={cov} ===")
    print("Genomy w tym pokryciu:", GENOMES)

    # --- akumulatory dla TEGO pokrycia (cov) ---

    sum_w_dec = 0.0   # suma wag decyzji
    sum_w_bub = 0.0   # suma wag bąbli

    sum_acc_dec = 0.0
    sum_prec    = 0.0
    sum_rec     = 0.0
    sum_f1      = 0.0

    sum_acc_bub = 0.0

    total_decisions = 0
    total_bubbles   = 0
    params = None

    for fold, test_genome in enumerate(GENOMES, start=1):
        test_paths  = [GENOME_TO_PATH[test_genome]]
        train_paths = [GENOME_TO_PATH[g] for g in GENOMES if g != test_genome]

        train_full = HyperbubbleDataset(train_paths)
        test_full  = HyperbubbleDataset(test_paths)

        train_lab = labeled_subset(train_full)
        test_lab  = labeled_subset(test_full)

        if len(train_lab) == 0 or len(test_lab) == 0:
            print(f"[WARN] brak etykiet dla test_genome={test_genome} (cov={cov}), pomijam fold")
            continue

        tr_idx, va_idx = split_train_val(len(train_lab), VAL_FRAC, SEED)
        train_ds = Subset(train_lab, tr_idx)
        val_ds   = Subset(train_lab, va_idx)
        test_ds  = test_lab

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

        print(
            f"[{variant}] coverage={cov} fold={fold} test={test_genome} "
            f"(train={len(train_ds)} val={len(val_ds)} test={len(test_ds)})"
        )

        for r in range(1, REPEATS + 1):
            # różne seed'y dla każdego (coverage, fold, repeat), żeby nie nachodziły na siebie
            run_seed = SEED + 1000 * cov + 100 * fold + r
            set_seed(run_seed)

            ckpt = OUTDIR / (
                f"{variant}_cov{cov}_fold{fold}_"
                f"{test_genome}_run{r}.pth"
            )

            best_state, _ = fit_variant(
                build_model_fn=build_model,
                train_loader=train_loader,
                val_loader=val_loader,
                device=device,
                out_ckpt=ckpt,
                epochs=EPOCHS,
                lr=LR,
                weight_decay=WEIGHT_DECAY,
                patience=PATIENCE,
                grad_clip=GRAD_CLIP,
                seed=run_seed,
                early_stop=True,
                print_every=50,
            )

            model = build_model().to(device)
            model.load_state_dict(best_state)
            model.eval()

            stats = eval_rich_metrics(model, test_loader, device)

            if params is None:
                params = count_params(model)

            # wagi
            w_dec = int(stats.get("decisions", 0))
            w_bub = int(stats.get("bubbles", 0))

            # akumulacja metryk ważonych liczbą decyzji
            if w_dec > 0:
                sum_w_dec += w_dec
                total_decisions += w_dec
                sum_acc_dec += stats["acc_dec"] * w_dec
                sum_prec    += stats["precision_dec"] * w_dec
                sum_rec     += stats["recall_dec"] * w_dec
                sum_f1      += stats["f1_dec"] * w_dec

            # akumulacja metryk ważonych liczbą bąbli
            if w_bub > 0:
                sum_w_bub += w_bub
                total_bubbles += w_bub
                sum_acc_bub += stats["acc_bub"] * w_bub

            print(
                f"[RUN {r}/{REPEATS}] cov={cov} {variant} | fold={fold} | test={test_genome} -> "
                f"acc_dec={stats['acc_dec']:.4f} "
                f"prec={stats['precision_dec']:.4f} "
                f"rec={stats['recall_dec']:.4f} "
                f"f1={stats['f1_dec']:.4f} "
                f"acc_bub={stats['acc_bub']:.4f} "
                f"(decisions={w_dec}, bubbles={w_bub})"
            )

    # --- finalizacja metryk dla TEGO pokrycia (cov) ---

    if sum_w_dec > 0:
        acc_dec = sum_acc_dec / sum_w_dec
        precision_dec = sum_prec / sum_w_dec
        recall_dec = sum_rec / sum_w_dec
        f1_dec = sum_f1 / sum_w_dec
    else:
        acc_dec = precision_dec = recall_dec = f1_dec = float("nan")

    if sum_w_bub > 0:
        acc_bub = sum_acc_bub / sum_w_bub
    else:
        acc_bub = float("nan")

    summary_row = pd.DataFrame([{
        "variant": variant,
        "coverage": cov,
        "runs": REPEATS,
        "params": params,
        "acc_dec": acc_dec,
        "precision_dec": precision_dec,
        "recall_dec": recall_dec,
        "f1_dec": f1_dec,
        "acc_bub": acc_bub,
        "decisions": int(total_decisions),
        "bubbles": int(total_bubbles),
    }])

    summary_row.to_csv(SUMMARY_CSV, mode="a", header=False, index=False)

    print(
        f"[SUMMARY] coverage={cov}, {variant} -> "
        f"acc_dec={acc_dec:.4f} prec={precision_dec:.4f} rec={recall_dec:.4f} "
        f"f1={f1_dec:.4f} acc_bub={acc_bub:.4f} "
        f"(decisions={total_decisions}, bubbles={total_bubbles})"
    )

print(f"\nSaved coverage-sweep results to: {SUMMARY_CSV}")


Uruchamianie eksperymentu coverage-sweep dla wariantu: emb8_g16_e256_mean

=== COVERAGE SWEEP: coverage=10 ===
Genomy w tym pokryciu: ['ecoli_dataset_k21_cov10_ratio_lt_3.5.jsonl', 'klebsiella_dataset_k21_cov10_ratio_lt_3.5.jsonl', 'proteus_dataset_k21_cov10_ratio_lt_3.5.jsonl']
[emb8_g16_e256_mean] coverage=10 fold=1 test=ecoli_dataset_k21_cov10_ratio_lt_3.5.jsonl (train=538 val=60 test=414)
[ep 001] train_loss=0.7172  val_bub=0.6333  val_dec=0.6129
[ep 050] train_loss=0.4164  val_bub=0.6500  val_dec=0.6613
[ep 100] train_loss=0.3909  val_bub=0.7667  val_dec=0.7742
[ep 150] train_loss=0.3477  val_bub=0.7167  val_dec=0.7258
[ep 200] train_loss=0.3261  val_bub=0.7000  val_dec=0.7097
[RUN 1/5] cov=10 emb8_g16_e256_mean | fold=1 | test=ecoli_dataset_k21_cov10_ratio_lt_3.5.jsonl -> acc_dec=0.6828 prec=0.6828 rec=1.0000 f1=0.8115 acc_bub=0.6812 (decisions=435, bubbles=414)
[ep 001] train_loss=0.7586  val_bub=0.6667  val_dec=0.6452
[ep 050] train_loss=0.4237  val_bub=0.7167  val_dec=0.7258
[

In [20]:
import pandas as pd
import matplotlib.pyplot as plt

# --- 0. Wczytanie wyników z eksperymentu pokryć ---

results_path = OUTDIR / "coverage_sweep_results.csv"
df = pd.read_csv(results_path)

print("Wczytano", len(df), "wierszy z", results_path)

# --- 1. Agregacja po (variant, coverage) ---

agg = (
    df.groupby(["variant", "coverage"], as_index=False)
      .agg(
          acc_dec_mean=("acc_dec", "mean"),
          precision_dec_mean=("precision_dec", "mean"),
          recall_dec_mean=("recall_dec", "mean"),
          f1_dec_mean=("f1_dec", "mean"),
          params=("params", "first"),
          decisions=("decisions", "sum"),
          bubbles=("bubbles", "sum"),
      )
      .sort_values(["variant", "coverage"])
)

agg_path = OUTDIR / "coverage_sweep_summary.csv"
agg.to_csv(agg_path, index=False)
print("Zapisano podsumowanie pokryć:", agg_path)

# --- 2. Wykres: dokładność decyzji vs pokrycie (osobno dla każdego wariantu) ---

# Jeśli masz tylko jeden wariant, będzie jedna linia; jeśli więcej – kilka linii na jednym wykresie.
plt.figure(figsize=(6, 4))

for variant, sub in agg.groupby("variant"):
    sub_sorted = sub.sort_values("coverage")
    plt.plot(
        sub_sorted["coverage"],
        sub_sorted["acc_dec_mean"],
        marker="o",
        label=str(variant),
    )

plt.xlabel("Pokrycie (coverage)")
plt.ylabel("Dokładność decyzji (średnia)")
plt.title("Dokładność decyzji vs pokrycie")
plt.legend(loc="best")
plt.tight_layout()

cov_plot_path = OUTDIR / "coverage_vs_acc_dec.png"
plt.savefig(cov_plot_path, dpi=200)
plt.close()
print("Zapisano wykres coverage vs acc_dec:", cov_plot_path)

# --- 3. Info tekstowe o najlepszym pokryciu (wg acc_dec_mean) ---

best_row = agg.loc[agg["acc_dec_mean"].idxmax()]
print(
    "Najlepsze pokrycie (wg acc_dec_mean): "
    f"variant={best_row['variant']}, coverage={best_row['coverage']}, "
    f"acc_dec_mean={best_row['acc_dec_mean']:.4f}"
)

print("Wyniki eksperymentu z pokryciami zapisane w katalogu:", OUTDIR.resolve())


Wczytano 3 wierszy z coverage_sweep_28_12\coverage_sweep_results.csv
Zapisano podsumowanie pokryć: coverage_sweep_28_12\coverage_sweep_summary.csv
Zapisano wykres coverage vs acc_dec: coverage_sweep_28_12\coverage_vs_acc_dec.png
Najlepsze pokrycie (wg acc_dec_mean): variant=emb8_g16_e256_mean, coverage=10, acc_dec_mean=0.7072
Wyniki eksperymentu z pokryciami zapisane w katalogu: C:\Users\Przemek\Desktop\Inzynierka stuff\own_assembler\dna-assembly-bsc\bubble_resolution_gnn\coverage_sweep_28_12
